<a href="https://colab.research.google.com/github/Fanfan0315/ESE5971-project/blob/main/5971_second_report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#install package
repos_url <- "https://cloud.r-project.org"

install.packages("vctrs", repos = repos_url, dependencies = TRUE)

pkgs <- c("readxl", "knitr", "kableExtra", "dplyr", "tidyr")

install.packages(pkgs, repos = repos_url, dependencies = TRUE)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘lazyeval’, ‘rex’, ‘covr’, ‘zeallot’


Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘AsioHeaders’, ‘R.methodsS3’, ‘R.oo’, ‘R.utils’, ‘websocket’, ‘XML’, ‘rjson’, ‘RcppTOML’, ‘here’, ‘sysfonts’, ‘showtextdb’, ‘R.cache’, ‘base64url’, ‘igraph’, ‘secretbase’, ‘filehash’, ‘chromote’, ‘formatR’, ‘gifski’, ‘gridSVG’, ‘jpeg’, ‘JuliaCall’, ‘magick’, ‘litedown’, ‘markdown’, ‘otelsdk’, ‘png’, ‘reticulate’, ‘rgl’, ‘showtext’, ‘styler’, ‘targets’, ‘testit’, ‘tikzDevice’, ‘webshot’, ‘svglite’, ‘formattable’, ‘sparkline’, ‘webshot2’, ‘Lahman’, ‘lobstr’, ‘nycflights13’, ‘RSQLite’, ‘repurrrsive’


Warning message in install.packages(pkgs, repos = repos_url, dependencies = TRUE):
“installation of package ‘gifski’ had non-zero exit status”
Warning message in install.packages(pkgs, repos = repos_url, dependencies = TRUE)

In [ ]:
#input dataset
library(readxl)
df <- read_excel("/Telco_customer_churn.xlsx")

ERROR: Error: `path` does not exist: ‘/Telco_customer_churn.xlsx’


In [ ]:
#2.1.1 dataset overview、
head(df)
str(df)
dim(df)

In [ ]:
#make table 1.Variable Classification by Data Type
library(knitr)
library(kableExtra)

#identify variables
numerical_vars <- names(df)[sapply(df, is.numeric)]
categorical_vars <- names(df)[!sapply(df, is.numeric)]

#combine variable names into single strings
table_type <- data.frame(
  Data_Type = c("Categorical", "Numerical"),
  Variables = c(
    paste(categorical_vars, collapse = ", "),
    paste(numerical_vars, collapse = ", ")
  )
)
#generate table
kable(table_type, caption = "Variable Classification by Data Type")

In [ ]:
#make table 2. Variable Classification by Business Domain
library(knitr)

demographic_vars <- c("Gender","SeniorCitizen","Partner","Dependents")

service_vars <- c(
  "PhoneService","MultipleLines","InternetService",
  "OnlineSecurity","OnlineBackup","DeviceProtection",
  "TechSupport","StreamingTV","StreamingMovies"
)

account_fin_vars <- c(
  "TenureMonths","Contract","PaperlessBilling",
  "PaymentMethod","MonthlyCharges","TotalCharges","CLTV"
)

table_domain <- data.frame(
  Category = c(
    "Demographic Characteristics",
    "Service Subscription Details",
    "Account and Financial Information"
  ),
  Variables = c(
    paste(demographic_vars, collapse = ", "),
    paste(service_vars, collapse = ", "),
    paste(account_fin_vars, collapse = ", ")
  )
)

kable(table_domain, caption = "Variable Classification by Business Domain")

In [ ]:
#2.1.2check target variable distribution
library(knitr)

target_dist <- data.frame(
  ChurnValue = names(table(df[["Churn Value"]])),
  Count = as.vector(table(df[["Churn Value"]])),
  Proportion = as.vector(prop.table(table(df[["Churn Value"]])))
)

kable(target_dist, digits = 3,
      caption = "Target Variable Distribution -- Churn Value")

In [ ]:
#2.1.3 numerical feature analysis
library(dplyr)
library(tidyr)
library(knitr)

num_vars_clean <- c(
  "Tenure Months",
  "Monthly Charges",
  "Total Charges",
  "Churn Score",
  "CLTV"
)

desc_table <- df %>%
  select(all_of(num_vars_clean)) %>%
  summarise(across(
    everything(),
    list(
      Mean = ~mean(., na.rm = TRUE),
      SD   = ~sd(., na.rm = TRUE),
      Min  = ~min(., na.rm = TRUE),
      Max  = ~max(., na.rm = TRUE)
    )
  )) %>%
  pivot_longer(
    cols = everything(),
    names_to = c("Variable", ".value"),
    names_sep = "_"
  )

kable(desc_table, digits = 2,
      caption = "Descriptive Statistics of Numerical Variables")

In [ ]:
#2.1.4 Categorical Feature Analysis
#=================Contract vs Churn==================
contract_churn <- table(df_clean$Contract, df_clean$`Churn Label`)
barplot(contract_churn,
        beside = TRUE,
        col = c("skyblue", "salmon"),
        legend = TRUE,
        main = "Churn by Contract Type",
        xlab = "Contract Type",
        ylab = "Number of Customers")

#=================Internet Service vs Churn==================
internet_churn <- table(df_clean$`Internet Service`, df_clean$`Churn Label`)
barplot(internet_churn,
        beside = TRUE,
        col = c("lightgreen", "orange"),
        legend = TRUE,
        main = "Churn by Internet Service",
        xlab = "Internet Service Type",
        ylab = "Number of Customers")

#=================Payment Method vs Churn=================
payment_churn <- table(df_clean$`Payment Method`, df_clean$`Churn Label`)
barplot(payment_churn,
        beside = TRUE,
        col = c("lightblue", "pink"),
        legend = TRUE,
        main = "Churn by Payment Method",
        xlab = "Payment Method",
        ylab = "Number of Customers")

In [ ]:
# 2.1.5 Bivariate Relationships with Churn
#==================Numerical Variables vs Churn==================
# Tenure Months vs Churn
boxplot(df_clean$`Tenure Months` ~ df_clean$`Churn Label`,
        col = c("lightblue","salmon"),
        main = "Tenure Months vs Churn",
        xlab = "Churn",
        ylab = "Tenure Months")

# Monthly Charges vs Churn
boxplot(df_clean$`Monthly Charges` ~ df_clean$`Churn Label`,
        col = c("lightgreen","orange"),
        main = "Monthly Charges vs Churn",
        xlab = "Churn",
        ylab = "Monthly Charges")

# Total Charges vs Churn
boxplot(df_clean$`Total Charges` ~ df_clean$`Churn Label`,
        col = c("pink","skyblue"),
        main = "Total Charges vs Churn",
        xlab = "Churn",
        ylab = "Total Charges")


#==================Categorical Variables vs Churn==================
# Contract vs Churn Rate
contract_rate <- prop.table(table(df_clean$Contract, df_clean$`Churn Label`),1)
barplot(contract_rate[,2],
        col = "tomato",
        main = "Churn Rate by Contract Type",
        xlab = "Contract Type",
        ylab = "Churn Rate")


# Internet Service vs Churn Rate
internet_rate <- prop.table(table(df_clean$`Internet Service`, df_clean$`Churn Label`),1)
barplot(internet_rate[,2],
        col = "steelblue",
        main = "Churn Rate by Internet Service",
        xlab = "Internet Service",
        ylab = "Churn Rate")


# Payment Method vs Churn Rate
payment_rate <- prop.table(table(df_clean$`Payment Method`, df_clean$`Churn Label`),1)
barplot(payment_rate[,2],
        col = "purple",
        las = 2,
        main = "Churn Rate by Payment Method",
        xlab = "Payment Method",
        ylab = "Churn Rate")

In [ ]:
#2.2.1.a identify outlier
#=======================IQR Method=======================
boxplot(df$`Total Charges`,
        main="Boxplot of Total Charges",
        ylab="Charges")

df$Total.Charges_log <- log(df$`Total Charges` + 1)

# Compute Q1 & Q3
Q1 = quantile(df$`Total Charges`, 0.25, na.rm = TRUE)
Q3 = quantile(df$`Total Charges`, 0.75, na.rm = TRUE)

# Compute IQR
IQR_value = Q3 - Q1

# Lower and upper bound
lower_bound = Q1 - 1.5 * IQR_value
upper_bound = Q3 + 1.5 * IQR_value

# Extract outliers
outliers_TotalCharges =
df$`Total Charges`[
df$`Total Charges` < lower_bound |
df$`Total Charges` > upper_bound
]

# Print count
print(paste("Total Charges outlier count:", length(outliers_TotalCharges)))

missing_data = df[is.na(df$`Total Charges`), ]

real_outliers = outliers_TotalCharges[!is.na(outliers_TotalCharges)]

print(paste("Real Total Charges outliers:", length(real_outliers)))

In [ ]:
#2.2.1.a identify outlier
#=======================Z-score Method=======================


x <- df$`Total Charges`

z_scores <- (x - mean(x, na.rm=TRUE)) / sd(x, na.rm=TRUE)

outliers_z <- x[abs(z_scores) > 3]

print(paste("Z-score outliers:", length(outliers_z)))

In [ ]:
#2.2.1.a identify outlier
#=======================Boxplot Method=======================
box <- boxplot(df$`Total Charges`, plot = FALSE)

outliers_box <- box$out

print(paste("Boxplot outliers:", length(outliers_box)))

In [ ]:
#2.2.1.b identify missing value
missing_summary <- data.frame(
  Variable = names(df),
  Missing_Count = colSums(is.na(df)),
  Missing_Percentage = colSums(is.na(df))/nrow(df)*100
)

kable(missing_summary)

In [ ]:
#2.2.2.b missing value treatment
df_clean <- df

# Replace with the median value
df_clean$`Total Charges`[is.na(df_clean$`Total Charges`)] <-
median(df_clean$`Total Charges`, na.rm = TRUE)

# Check the number of missing values
colSums(is.na(df_clean))